# kelv_helm2: Kelvin-Helmholtz instability of an unstratified vortex sheet
pyOM2: py_config/kelv_helm2.py

Non-hydrostatic x-z slice (nh_simple). A thin shear layer in a homogeneous fluid,
perturbed with the streamfunction of the linear instability.

In [ ]:
import sys; sys.path.append('../..')
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import pyOM3 as OM
from pyOM3.nh_simple.model import model

In [ ]:
fac        = 1.0
nx, ny, nz = int(1.5*64*fac), 1, int(40*fac)
dx = dz    = 0.25/fac
dt         = 0.04/fac
mix        = 2.5e-3/fac**2
Lx, Lz     = nx*dx, nz*dz
kx         = 4*np.pi/Lx             # wavenumber of the perturbation
amp        = 0.2                    # amplitude of the perturbation
nsteps     = 1500                   # 60 s
plot_every = 100

par = dict(nx=nx, ny=ny, nz=nz, dx=dx, dy=dx, dz=dz, dt=dt,
           f0=0., N0=0., Ro=1., dsqr=1., periodic_in_z=False,
           Ah=mix, Av=mix, buoyancy_superbee_advection=True)

In [ ]:
M = model(par)
M.setup()
# local coordinates with halo points, arrays are (x,y,z), z = 0 at the surface
h = M.halo
x = (np.arange(M.Nx) - h + M.xs_pe - 0.5)*dx
y = (np.arange(M.Ny) - h + M.ys_pe - 0.5)*dx
z = (np.arange(M.Nz) - h + M.zs_pe - 0.5)*dz - nz*dz
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
z_star = -Lz/2
u = 0.6 - 0.5*np.tanh((Z - z_star)/Lz*300)
# streamfunction of the perturbation, decaying away from the interface
psi = np.where(Z > z_star, amp*np.exp(-kx*(Z - z_star))*np.exp(1j*kx*X),
               -1j*amp*np.exp(kx*(Z - z_star))*np.exp(1j*kx*X))
u = u - np.real(np.where(Z > z_star, -kx*psi, kx*psi))
w = np.real(1j*kx*psi)
M.u = M.apply_bc(OM.np.asarray(u))
M.w = M.apply_vertical_flux_bc(M.apply_bc(OM.np.asarray(w)))

In [ ]:
def plot(t):
    clear_output(wait=True)
    u, w = M.u[h:-h, h, h:-h].T, M.w[h:-h, h, h:-h].T
    vort = np.gradient(u, dz, axis=0) - np.gradient(w, dx, axis=1)
    plt.figure(figsize=(9, 4))
    plt.pcolormesh(x[h:-h], z[h:-h], vort, cmap='RdBu_r', vmin=-2, vmax=2)
    plt.colorbar(label='vorticity [1/s]'); plt.xlabel('x [m]'); plt.ylabel('z [m]')
    plt.title('t = %.1f s' % t); plt.show()

In [ ]:
A, B, C = 1., 0., 0.                 # Adams-Bashforth weights, forward Euler at the start
tau, taum1, taum2 = 2, 1, 0          # pointers to the time levels
for n in range(nsteps):
    M.time_step(A, B, C, tau, taum1, taum2)
    if n % plot_every == 0: plot(n*dt)
    tau, taum1, taum2 = (tau+1) % 3, (taum1+1) % 3, (taum2+1) % 3
    A, B, C = (1.5, -0.5, 0.) if n == 0 else (23./12, -16./12, 5./12)

In [ ]:
plot(nsteps*dt)